# 3.2 Binary Classification

使用 breast cancer dataset 建立二元分類 DNN。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report

## 1. 載入資料

In [ ]:
data = load_breast_cancer(as_frame=True)
df = data.frame

print(df.shape)
df.head()

## 2. 切分資料與標準化

In [ ]:
X = df.drop(columns=['target']).values
y = df['target'].values

x_train, x_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

print(x_train.shape, x_test.shape)

## 3. 建立模型

In [ ]:
tf.keras.utils.set_random_seed(42)

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(x_train.shape[1],)),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(16, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc'), tf.keras.metrics.Precision(name='precision'), tf.keras.metrics.Recall(name='recall')]
)

model.summary()

## 4. 訓練模型

In [ ]:
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=10, restore_best_weights=True
)

history = model.fit(
    x_train, y_train,
    validation_split=0.2,
    epochs=100,
    batch_size=16,
    callbacks=[early_stop],
    verbose=0
)

pd.DataFrame(history.history)[['loss', 'val_loss']].plot(figsize=(8, 4), title='Loss Curve')
plt.show()

## 5. 評估模型

In [ ]:
train_result = model.evaluate(x_train, y_train, verbose=0, return_dict=True)
test_result = model.evaluate(x_test, y_test, verbose=0, return_dict=True)

result_df = pd.DataFrame({
    'metric': list(train_result.keys()),
    'train': list(train_result.values()),
    'test': list(test_result.values())
})
result_df

In [ ]:
y_prob = model.predict(x_test, verbose=0).ravel()
y_pred = (y_prob >= 0.5).astype(int)

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=data.target_names))

## 6. 預測新資料

In [ ]:
sample = x_test[:5]
prob = model.predict(sample, verbose=0).ravel()
pd.DataFrame({'probability_positive_class': prob, 'predicted_label': (prob >= 0.5).astype(int), 'true_label': y_test[:5]})

## 7. 套用自己的資料

將自己的資料整理成 `X` 與 `y`，維持最後一層 `Dense(1, activation="sigmoid")`，loss 使用 `binary_crossentropy`。